<a href="https://colab.research.google.com/github/LamLam5406/IRAN-USA_Chatbot/blob/main/IRAN_USA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Cài đặt thư viện và thiết lập API

In [ ]:
# Cài đặt lại các thư viện với phiên bản ổn định và đầy đủ
!pip install -Uq langchain langchain-community langchain-google-genai chromadb langchain-huggingface

import langchain
print(f"Phiên bản LangChain đã cài đặt: {langchain.__version__}")

Phiên bản LangChain đã cài đặt: 1.3.11


In [ ]:
import os
from google.colab import userdata
import google.generativeai as genai

# Lấy API key từ bộ nhớ an toàn của Colab
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

# Cấu hình thư viện genai
genai.configure(api_key=GOOGLE_API_KEY)

print("Đã kết nối thành công với Gemini API!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Đã kết nối thành công với Gemini API!


# 2. Tải, Làm sạch và Chuẩn bị Dữ liệu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# 1. Đọc dữ liệu từ file CSV
# Thay đổi tên file nếu file của bạn có tên khác
file_path = '/content/drive/MyDrive/Projects/IRAN_USA/iran_us_news_dataset.csv'
df = pd.read_csv(file_path)

print(f"Tổng số bài báo ban đầu: {len(df)}")

# 2. Làm sạch dữ liệu
# Loại bỏ các hàng không có nội dung bài viết
df = df.dropna(subset=['content'])

# Xử lý các bài báo bị thiếu ngày tháng
df['date'] = df['date'].fillna('Không xác định')

# Xử lý các bài báo bị thiếu nguồn
df['_source'] = df['_source'].fillna('Nguồn ẩn danh')

# 3. Ghép Metadata vào nội dung
# Khởi tạo một cột mới chứa nội dung đã được làm giàu thông tin
df['enriched_content'] = (
    "[Nguồn: " + df['_source'] + " | " +
    "Ngày: " + df['date'].astype(str) + " | " +
    "Tiêu đề: " + df['title'] + "]\n\n" +
    df['content']
)

print(f"Tổng số bài báo sau khi làm sạch: {len(df)}")

# Xem thử một đoạn dữ liệu mẫu đã được xử lý
print("\n--- MẪU DỮ LIỆU ĐÃ XỬ LÝ ---")
print(df['enriched_content'].iloc[0][:500] + "...\n")

Tổng số bài báo ban đầu: 2286
Tổng số bài báo sau khi làm sạch: 2286

--- MẪU DỮ LIỆU ĐÃ XỬ LÝ ---
[Nguồn: Bing-Al Jazeera | Ngày: 2026-06-13T00:00:00 | Tiêu đề: As Iran and US near a deal, Tehran remembers another recent bloody conflict]

Iranian authorities say assassinations and strikes over the past year have failed to deter them.

Tehran, Iran – The anniversary of a 12-day war between Iran and Israel in June 2025 is being marked this week in Tehran, as American and Iranian officials engage in last-minute negotiations to end a more recent conflict between the two sides.

Tehran and Washin...



# 3. Cắt nhỏ văn bản và Tạo Vector Database

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Đang tiến hành cắt nhỏ văn bản (Chunking)...")

# 1. Khởi tạo bộ cắt văn bản
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = df['enriched_content'].tolist()
chunks = text_splitter.create_documents(documents)
print(f"Đã cắt thành công {len(chunks)} đoạn nhỏ từ {len(df)} bài báo.")

# 2. Sử dụng mô hình Embedding cục bộ
print("\nĐang tải mô hình Embedding cục bộ (chỉ tốn vài giây)...")
# 'all-MiniLM-L6-v2' là mô hình cực nhẹ, chuẩn xác cho tiếng Anh và chạy mượt trên CPU
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. Lưu cơ sở dữ liệu vector vào Google Drive
drive_chroma_path = '/content/drive/MyDrive/Projects/IRAN_USA/chroma_db_storage'

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=drive_chroma_path # Thay đổi ở đây
)

print("Hệ thống đã học xong toàn bộ bộ dữ liệu và lưu vào Vector DB.")

/tmp/ipykernel_22042/3704915242.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


Đang tiến hành cắt nhỏ văn bản (Chunking)...
Đã cắt thành công 10011 đoạn nhỏ từ 2286 bài báo.

Đang tải mô hình Embedding cục bộ (chỉ tốn vài giây)...


/tmp/ipykernel_22042/3704915242.py:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Hệ thống đã học xong toàn bộ bộ dữ liệu và lưu vào Vector DB.


# 4. Tạo Luồng Hỏi Đáp (RAG Chain)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("Đang thiết lập luồng RAG theo kiến trúc cốt lõi...")

# 1. Khởi tạo mô hình Gemini
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.2)

# 2. Thiết lập Retriever lấy 10 đoạn văn bản
retriever = vector_db.as_retriever(search_kwargs={"k": 10})

# 3. Hàm phụ trợ: Gộp nội dung 10 đoạn văn bản lại thành 1 cục text dài
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Tạo Prompt Template
template = """Bạn là trợ lý chuyên gia phân tích địa chính trị về xung đột Iran - Mỹ (2025-2026).
Sử dụng các tài liệu được cung cấp dưới đây để trả lời câu hỏi của người dùng.
Nếu thông tin không có trong tài liệu, hãy thẳng thắn nói rằng bạn không biết, đừng tự bịa ra.

TÀI LIỆU CUNG CẤP:
{context}

Câu hỏi: {question}
Câu trả lời:"""

prompt = ChatPromptTemplate.from_template(template)

# 5. Lắp ráp chuỗi bằng LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() # Tự động bóc tách text trả về, không bị dính định dạng rườm rà
)

print("Luồng RAG đã lắp ráp xong! Sẵn sàng.")

Đang thiết lập luồng RAG theo kiến trúc cốt lõi...
Luồng RAG đã lắp ráp xong! Sẵn sàng.


In [ ]:
query = "Chiến dịch Midnight Hammer là gì và diễn ra khi nào?"

# Vì rag_chain được thiết kế với RunnablePassthrough() nhận query trực tiếp,
# chúng ta truyền thẳng chuỗi 'query' vào thay vì dùng dict.
response = rag_chain.invoke(query)

print("KẾT QUẢ TỪ BOT RAG:\n")
print(response)

KẾT QUẢ TỪ BOT RAG:

Dựa trên các tài liệu được cung cấp, thông tin về chiến dịch Midnight Hammer được xác định như sau:

*   **Chiến dịch Midnight Hammer là gì:** 
    *   Đây là một chiến dịch tấn công quân sự của Mỹ nhắm vào Iran, cụ thể là nhằm vào 3 cơ sở hạt nhân của nước này.
    *   Đây là một sứ mệnh tuyệt mật, phức tạp và có độ rủi ro cao, đòi hỏi kỹ năng và kỷ luật của lực lượng liên quân. 
    *   Mục tiêu của chiến dịch là phá hủy nghiêm trọng cơ sở hạ tầng vũ khí hạt nhân của Iran và tập trung vào việc ngăn chặn phổ biến vũ khí hạt nhân.
    *   Chiến dịch này diễn ra trong thời gian ngắn với mô hình tấn công chính xác (khác với chiến dịch Epic Fury là một chiến dịch kéo dài nhiều ngày).

*   **Diễn ra khi nào:** 
    *   Tài liệu không nêu rõ ngày giờ cụ thể cuộc tấn công thực tế được thực hiện. Tuy nhiên, tài liệu cho biết vào **Chủ Nhật, ngày 23 tháng 6 năm 2025**, Lầu Năm Góc (Pentagon) đã công bố chi tiết về cuộc tấn công có chủ đích này.


# 5. Tạo Giao Diện Chat Trực Quan với Gradio

In [ ]:
import gradio as gr

def respond(message, history):
    try:
        # Gọi RAG lấy câu trả lời
        bot_message = rag_chain.invoke(message)
        return bot_message

    except Exception as e:
        # Nếu lỗi giới hạn token hoặc mạng, in thẳng ra khung chat
        error_msg = str(e)
        if "429" in error_msg or "Exhausted" in error_msg:
            return "⏳ API đang bị quá tải giới hạn miễn phí (15 câu/phút). Bạn đợi khoảng 1 phút rồi hỏi lại nhé!"
        else:
            return f"⚠️ Có lỗi xảy ra: {error_msg}"

# Giao diện nguyên bản
demo = gr.ChatInterface(
    fn=respond,
    title="Trợ Lý Ảo Phân Tích Xung Đột Iran - Mỹ (2025-2026)",
    description="Mọi thông tin đều có khả năng sai sót, cân nhắc kỹ trước khi lan truyền.",
    examples=[
        "Chiến dịch Midnight Hammer là gì?",
        "Chuyện gì đã xảy ra tại cơ sở hạt nhân Fordow?",
        "Fox News và Al Jazeera đưa tin khác nhau thế nào?"
    ]
)

# Khởi chạy
demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://14698dca1d1c7e9c82.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://14698dca1d1c7e9c82.gradio.live
